# MLP 하이퍼파라미터 튜닝

저장된 Train 기준 전처리기를 재사용해 MLP의 클래스 가중치를 비교한다.

원칙:

- 기존 MLP와 동일한 Train·Valid 데이터 사용
- 기존 scaler와 encoder 재사용
- Test 데이터는 튜닝 과정에서 사용하지 않음
- Valid PR-AUC로 최적 설정 선택
- 첫 실험은 클래스 가중치 14.64를 약 3.83으로 완화

In [2]:
import copy
import gc
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)


CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [
            CURRENT_DIR,
            *CURRENT_DIR.parents,
        ]
        if (
            (path / "preprocessing").is_dir()
            and (path / "modeling").is_dir()
            and (path / ".gitignore").exists()
        )
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "수정본 프로젝트 경로를 찾을 수 없습니다."
    )

PROCESSED_DIR = (
    PROJECT_ROOT / "data" / "processed"
)

MODELS_DIR = PROJECT_ROOT / "models"

print("현재 경로:", CURRENT_DIR)
print("프로젝트 루트:", PROJECT_ROOT)

현재 경로: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\dl_modeling
프로젝트 루트: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify


In [ ]:
# 기존 전처리기와 기준 성능 불러오기
import json

PREPROCESSOR_PATH = (
    MODELS_DIR
    / "mlp_20170131_preprocessor.joblib"
)

BASELINE_META_PATH = (
    MODELS_DIR
    / "mlp_20170131_meta.json"
)

preprocessor = joblib.load(
    PREPROCESSOR_PATH
)

scaler = preprocessor["scaler"]
encoder = preprocessor["encoder"]

NUMERIC_COLS = preprocessor[
    "numeric_cols"
]

CATEGORICAL_COLS = preprocessor[
    "categorical_cols"
]

FINAL_FEATURE_NAMES = preprocessor[
    "feature_names"
]

with open(
    BASELINE_META_PATH,
    encoding="utf-8",
) as file:
    baseline_meta = json.load(file)

BASELINE_VALID_PR_AUC = baseline_meta[
    "valid_pr_auc"
]

print("수치형 피처:", len(NUMERIC_COLS))
print("범주형 피처:", len(CATEGORICAL_COLS))
print(
    "최종 입력 피처:",
    len(FINAL_FEATURE_NAMES),
)
print(
    "기존 MLP Valid PR-AUC:",
    f"{BASELINE_VALID_PR_AUC:.5f}",
)

수치형 피처: 37
범주형 피처: 3
최종 입력 피처: 68
기존 MLP Valid PR-AUC: 0.55186


In [ ]:
# Train·Valid 데이터 변환
DATA_PATH = (
    PROCESSED_DIR
    / "model_table_final.csv"
)

df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
)

assert df["snapshot"].eq(
    "2017-01-31"
).all()

assert df.isna().sum().sum() == 0

assert df["msno"].duplicated().sum() == 0

assert (
    df["days_since_last_txn"] < 0
).sum() == 0


train_df = df[
    df["split"] == "train"
].copy()

valid_df = df[
    df["split"] == "valid"
].copy()

test_count = (
    df["split"] == "test"
).sum()


X_train_numeric = scaler.transform(
    train_df[NUMERIC_COLS]
).astype(np.float32)

X_valid_numeric = scaler.transform(
    valid_df[NUMERIC_COLS]
).astype(np.float32)

X_train_categorical = encoder.transform(
    train_df[CATEGORICAL_COLS]
).astype(np.float32)

X_valid_categorical = encoder.transform(
    valid_df[CATEGORICAL_COLS]
).astype(np.float32)


X_train = np.hstack([
    X_train_numeric,
    X_train_categorical,
]).astype(np.float32)

X_valid = np.hstack([
    X_valid_numeric,
    X_valid_categorical,
]).astype(np.float32)


y_train = train_df[
    "is_churn"
].to_numpy(dtype=np.float32)

y_valid = valid_df[
    "is_churn"
].to_numpy(dtype=np.float32)


assert X_train.shape[1] == 68
assert X_valid.shape[1] == 68
assert np.isfinite(X_train).all()
assert np.isfinite(X_valid).all()

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("Test 미사용:", f"{test_count:,}")
print(
    "Train 이탈률:",
    f"{y_train.mean():.4f}",
)
print(
    "Valid 이탈률:",
    f"{y_valid.mean():.4f}",
)

X_train: (695051, 68)
X_valid: (148940, 68)
Test 미사용: 148,940
Train 이탈률: 0.0639
Valid 이탈률: 0.0639


In [5]:
# 메모리 정리 및 DataLoader
del (
    df,
    train_df,
    valid_df,
    X_train_numeric,
    X_valid_numeric,
    X_train_categorical,
    X_valid_categorical,
)

gc.collect()


RANDOM_SEED = 42
BATCH_SIZE = 4096

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

X_train_tensor = torch.from_numpy(
    X_train
)

X_valid_tensor = torch.from_numpy(
    X_valid
)

y_train_tensor = torch.from_numpy(
    y_train
).reshape(-1, 1)

y_valid_tensor = torch.from_numpy(
    y_valid
).reshape(-1, 1)


train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor,
)

valid_dataset = TensorDataset(
    X_valid_tensor,
    y_valid_tensor,
)


generator = torch.Generator()
generator.manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=False,
    generator=generator,
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))
print("튜닝 데이터 준비 완료")

Train: 695051
Valid: 148940
Train batches: 170
Valid batches: 37
튜닝 데이터 준비 완료


In [14]:
# 모델 구조와 장치
class ChurnMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.network(x)


device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

INPUT_DIM = X_train.shape[1]

negative_count = int(
    (y_train == 0).sum()
)

positive_count = int(
    (y_train == 1).sum()
)

full_positive_weight = (
    negative_count / positive_count
)

sqrt_positive_weight = np.sqrt(
    full_positive_weight
)

print("사용 장치:", device)
print(
    "기존 전체 가중치:",
    f"{full_positive_weight:.4f}",
)
print(
    "실험용 완화 가중치:",
    f"{sqrt_positive_weight:.4f}",
)

사용 장치: cpu
기존 전체 가중치: 14.6437
실험용 완화 가중치: 3.8267


In [15]:
# 평가함수
def evaluate_loader(
    model,
    data_loader,
    criterion,
):
    model.eval()

    total_loss = 0.0
    probabilities = []
    targets = []

    with torch.no_grad():
        for batch_X, batch_y in data_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            logits = model(batch_X)
            loss = criterion(logits, batch_y)

            total_loss += (
                loss.item() * len(batch_X)
            )

            probabilities.append(
                torch.sigmoid(logits)
                .cpu()
                .numpy()
                .reshape(-1)
            )

            targets.append(
                batch_y
                .cpu()
                .numpy()
                .reshape(-1)
            )

    probabilities = np.concatenate(
        probabilities
    )

    targets = np.concatenate(targets)

    return {
        "loss": (
            total_loss
            / len(data_loader.dataset)
        ),
        "auc": roc_auc_score(
            targets,
            probabilities,
        ),
        "pr_auc": average_precision_score(
            targets,
            probabilities,
        ),
        "probabilities": probabilities,
        "targets": targets,
    }

In [16]:
# 학습함수
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
MAX_EPOCHS = 30
PATIENCE = 5
MIN_DELTA = 1e-5


def train_mlp_experiment(
    experiment_name,
    pos_weight_value,
):
    # 매 실험을 동일한 초기 상태에서 시작
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)
    generator.manual_seed(RANDOM_SEED)

    experiment_model = ChurnMLP(
        input_dim=INPUT_DIM
    ).to(device)

    experiment_criterion = (
        nn.BCEWithLogitsLoss(
            pos_weight=torch.tensor(
                [pos_weight_value],
                dtype=torch.float32,
                device=device,
            )
        )
    )

    experiment_optimizer = torch.optim.AdamW(
        experiment_model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    history = []

    best_pr_auc = -np.inf
    best_epoch = 0
    best_state = None
    no_improvement_count = 0

    start_time = time.time()

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        epoch_start = time.time()

        experiment_model.train()
        total_train_loss = 0.0

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            experiment_optimizer.zero_grad(
                set_to_none=True
            )

            logits = experiment_model(batch_X)

            loss = experiment_criterion(
                logits,
                batch_y,
            )

            loss.backward()
            experiment_optimizer.step()

            total_train_loss += (
                loss.item() * len(batch_X)
            )

        train_loss = (
            total_train_loss
            / len(train_loader.dataset)
        )

        valid_result = evaluate_loader(
            model=experiment_model,
            data_loader=valid_loader,
            criterion=experiment_criterion,
        )

        epoch_seconds = (
            time.time() - epoch_start
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "valid_loss": valid_result["loss"],
            "valid_auc": valid_result["auc"],
            "valid_pr_auc": valid_result["pr_auc"],
            "seconds": epoch_seconds,
        })

        print(
            f"[{experiment_name}] "
            f"Epoch {epoch:02d}/{MAX_EPOCHS} | "
            f"Train Loss: {train_loss:.5f} | "
            f"Valid Loss: "
            f"{valid_result['loss']:.5f} | "
            f"Valid AUC: "
            f"{valid_result['auc']:.5f} | "
            f"Valid PR-AUC: "
            f"{valid_result['pr_auc']:.5f} | "
            f"{epoch_seconds:.1f}s"
        )

        if (
            valid_result["pr_auc"]
            > best_pr_auc + MIN_DELTA
        ):
            best_pr_auc = (
                valid_result["pr_auc"]
            )

            best_epoch = epoch

            best_state = copy.deepcopy(
                experiment_model.state_dict()
            )

            no_improvement_count = 0

        else:
            no_improvement_count += 1

            if (
                no_improvement_count
                >= PATIENCE
            ):
                print(
                    f"\n[{experiment_name}] "
                    "Early Stopping"
                )
                break

    experiment_model.load_state_dict(
        best_state
    )

    best_valid_result = evaluate_loader(
        model=experiment_model,
        data_loader=valid_loader,
        criterion=experiment_criterion,
    )

    elapsed_minutes = (
        time.time() - start_time
    ) / 60

    print(
        f"\n[{experiment_name}] 학습 완료"
    )
    print("Best Epoch:", best_epoch)
    print(
        "Best Valid AUC:",
        f"{best_valid_result['auc']:.5f}",
    )
    print(
        "Best Valid PR-AUC:",
        f"{best_valid_result['pr_auc']:.5f}",
    )
    print(
        "학습시간:",
        f"{elapsed_minutes:.1f}분",
    )

    return {
        "name": experiment_name,
        "model": experiment_model,
        "criterion": experiment_criterion,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "valid_result": best_valid_result,
        "pos_weight": pos_weight_value,
    }

In [17]:
# 완화된 클래스 가중치 실험
negative_count = int(
    (y_train == 0).sum()
)

positive_count = int(
    (y_train == 1).sum()
)

full_positive_weight = (
    negative_count / positive_count
)

sqrt_positive_weight = np.sqrt(
    full_positive_weight
)

print(
    "기존 전체 가중치:",
    f"{full_positive_weight:.4f}",
)

print(
    "새 완화 가중치:",
    f"{sqrt_positive_weight:.4f}",
)

기존 전체 가중치: 14.6437
새 완화 가중치: 3.8267


In [18]:
sqrt_weight_experiment = (
    train_mlp_experiment(
        experiment_name="sqrt_pos_weight",
        pos_weight_value=sqrt_positive_weight,
    )
)

[sqrt_pos_weight] Epoch 01/30 | Train Loss: 0.50010 | Valid Loss: 0.40113 | Valid AUC: 0.85516 | Valid PR-AUC: 0.43861 | 42.6s
[sqrt_pos_weight] Epoch 02/30 | Train Loss: 0.40144 | Valid Loss: 0.38473 | Valid AUC: 0.86425 | Valid PR-AUC: 0.49153 | 28.9s
[sqrt_pos_weight] Epoch 03/30 | Train Loss: 0.39073 | Valid Loss: 0.37650 | Valid AUC: 0.86930 | Valid PR-AUC: 0.52367 | 24.1s
[sqrt_pos_weight] Epoch 04/30 | Train Loss: 0.38444 | Valid Loss: 0.37260 | Valid AUC: 0.87366 | Valid PR-AUC: 0.52927 | 24.4s
[sqrt_pos_weight] Epoch 05/30 | Train Loss: 0.38072 | Valid Loss: 0.37042 | Valid AUC: 0.87628 | Valid PR-AUC: 0.53417 | 22.8s
[sqrt_pos_weight] Epoch 06/30 | Train Loss: 0.37859 | Valid Loss: 0.36869 | Valid AUC: 0.87784 | Valid PR-AUC: 0.53753 | 24.2s
[sqrt_pos_weight] Epoch 07/30 | Train Loss: 0.37683 | Valid Loss: 0.36766 | Valid AUC: 0.87992 | Valid PR-AUC: 0.54085 | 24.8s
[sqrt_pos_weight] Epoch 08/30 | Train Loss: 0.37503 | Valid Loss: 0.36632 | Valid AUC: 0.88119 | Valid PR-AUC: 

### 실험 1: 클래스 가중치 완화

기존에는 클래스 비율 전체를 반영한 `pos_weight=14.64`를 사용했다.
이번 실험에서는 과도한 가중치의 영향을 완화하기 위해 클래스 비율의
제곱근인 약 `3.83`을 적용했다.

실험 결과 Valid AUC는 거의 유지되었고,
Valid PR-AUC는 0.55186에서 0.55753으로 상승했다.
따라서 현재까지는 제곱근 클래스 가중치를 적용한 모델이 가장 우수하다.

In [19]:
no_weight_experiment = train_mlp_experiment(
    experiment_name="no_pos_weight",
    pos_weight_value=1.0,
)

[no_pos_weight] Epoch 01/30 | Train Loss: 0.32642 | Valid Loss: 0.18516 | Valid AUC: 0.84912 | Valid PR-AUC: 0.41564 | 51.2s
[no_pos_weight] Epoch 02/30 | Train Loss: 0.17515 | Valid Loss: 0.16323 | Valid AUC: 0.86000 | Valid PR-AUC: 0.46470 | 23.8s
[no_pos_weight] Epoch 03/30 | Train Loss: 0.16554 | Valid Loss: 0.15858 | Valid AUC: 0.86588 | Valid PR-AUC: 0.51025 | 24.3s
[no_pos_weight] Epoch 04/30 | Train Loss: 0.16187 | Valid Loss: 0.15628 | Valid AUC: 0.86881 | Valid PR-AUC: 0.52360 | 26.3s
[no_pos_weight] Epoch 05/30 | Train Loss: 0.15992 | Valid Loss: 0.15496 | Valid AUC: 0.87220 | Valid PR-AUC: 0.53052 | 24.8s
[no_pos_weight] Epoch 06/30 | Train Loss: 0.15892 | Valid Loss: 0.15413 | Valid AUC: 0.87416 | Valid PR-AUC: 0.53503 | 23.0s
[no_pos_weight] Epoch 07/30 | Train Loss: 0.15801 | Valid Loss: 0.15370 | Valid AUC: 0.87585 | Valid PR-AUC: 0.53872 | 23.6s
[no_pos_weight] Epoch 08/30 | Train Loss: 0.15734 | Valid Loss: 0.15317 | Valid AUC: 0.87730 | Valid PR-AUC: 0.54163 | 21.4s


### 클래스 가중치 실험 결과

세 가지 클래스 가중치를 비교한 결과, 가중치를 사용하지 않은
`pos_weight=1.0` 모델이 가장 높은 Valid PR-AUC 0.55979를 기록했다.

클래스 가중치를 낮출수록 Valid AUC는 소폭 하락했지만,
본 프로젝트의 주요 평가 지표인 Valid PR-AUC는 상승했다.
따라서 이후 튜닝에서는 `pos_weight=1.0`을 사용한다.

단, Test 데이터는 아직 사용하지 않고 모든 모델 선택은
Validation 성능만을 기준으로 진행한다.

## 최종 MLP 선택 및 Validation 임계값 결정

클래스 가중치 실험 중 Valid PR-AUC가 가장 높은
`pos_weight=1.0` 모델을 최종 MLP로 선택한다.

분류 임계값은 Test 데이터가 아니라 Validation 데이터에서
F1-Score가 최대가 되는 값으로 결정한다.

In [20]:
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

final_experiment = no_weight_experiment

valid_result = final_experiment["valid_result"]

valid_probabilities = valid_result[
    "probabilities"
]

valid_targets = valid_result[
    "targets"
].astype(int)

precision_curve, recall_curve, thresholds = (
    precision_recall_curve(
        valid_targets,
        valid_probabilities,
    )
)

f1_curve = (
    2
    * precision_curve[:-1]
    * recall_curve[:-1]
    / (
        precision_curve[:-1]
        + recall_curve[:-1]
        + 1e-12
    )
)

best_threshold_index = np.argmax(f1_curve)

final_threshold = float(
    thresholds[best_threshold_index]
)

valid_predictions = (
    valid_probabilities >= final_threshold
).astype(int)

print("최종 실험:", final_experiment["name"])
print(
    "Best Epoch:",
    final_experiment["best_epoch"],
)
print(
    "Valid AUC:",
    f"{valid_result['auc']:.5f}",
)
print(
    "Valid PR-AUC:",
    f"{valid_result['pr_auc']:.5f}",
)
print(
    "최적 Threshold:",
    f"{final_threshold:.5f}",
)
print(
    "Valid Precision:",
    f"{precision_score(valid_targets, valid_predictions):.5f}",
)
print(
    "Valid Recall:",
    f"{recall_score(valid_targets, valid_predictions):.5f}",
)
print(
    "Valid F1:",
    f"{f1_score(valid_targets, valid_predictions):.5f}",
)

최종 실험: no_pos_weight
Best Epoch: 29
Valid AUC: 0.88823
Valid PR-AUC: 0.55979
최적 Threshold: 0.29805
Valid Precision: 0.53937
Valid Recall: 0.54889
Valid F1: 0.54409


## Test 데이터 최종 평가

Validation 데이터에서 모델과 임계값을 확정한 후,
Test 데이터는 최종 일반화 성능 확인에 한 번만 사용한다.
학습할 때 저장한 전처리기를 그대로 적용한다.

In [21]:
test_df = pd.read_csv(
    DATA_PATH,
    low_memory=False,
)

test_df = test_df[
    test_df["split"] == "test"
].copy()

X_test_numeric = scaler.transform(
    test_df[NUMERIC_COLS]
).astype(np.float32)

X_test_categorical = encoder.transform(
    test_df[CATEGORICAL_COLS]
).astype(np.float32)

X_test = np.hstack([
    X_test_numeric,
    X_test_categorical,
]).astype(np.float32)

y_test = test_df[
    "is_churn"
].to_numpy(dtype=np.float32)

assert X_test.shape == (148940, 68)
assert np.isfinite(X_test).all()

test_dataset = TensorDataset(
    torch.from_numpy(X_test),
    torch.from_numpy(
        y_test.reshape(-1, 1)
    ),
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("X_test:", X_test.shape)
print("Test batches:", len(test_loader))
print("Test 데이터 준비 완료")

X_test: (148940, 68)
Test batches: 37
Test 데이터 준비 완료


In [22]:
test_result = evaluate_loader(
    model=final_experiment["model"],
    data_loader=test_loader,
    criterion=final_experiment["criterion"],
)

test_probabilities = test_result[
    "probabilities"
]

test_targets = test_result[
    "targets"
].astype(int)

test_predictions = (
    test_probabilities >= final_threshold
).astype(int)

print("===== 최종 MLP TEST 성능 =====")
print(
    "AUC:",
    f"{test_result['auc']:.5f}",
)
print(
    "PR-AUC:",
    f"{test_result['pr_auc']:.5f}",
)
print(
    "Precision:",
    f"{precision_score(test_targets, test_predictions):.5f}",
)
print(
    "Recall:",
    f"{recall_score(test_targets, test_predictions):.5f}",
)
print(
    "F1:",
    f"{f1_score(test_targets, test_predictions):.5f}",
)
print("\nConfusion Matrix:")
print(
    confusion_matrix(
        test_targets,
        test_predictions,
    )
)

===== 최종 MLP TEST 성능 =====
AUC: 0.89235
PR-AUC: 0.55990
Precision: 0.53514
Recall: 0.55273
F1: 0.54379

Confusion Matrix:
[[134849   4571]
 [  4258   5262]]


## 최종 MLP 모델 및 결과 저장

Validation PR-AUC를 기준으로 선택한 가중치 미적용 MLP의
가중치, 학습 기록, 평가 메타데이터 및 Test 예측 결과를 저장한다.

Test 데이터를 확인했으므로 이후에는 이 결과를 기준으로
추가 튜닝하지 않는다.

In [23]:
import json

MODELS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_PATH = (
    MODELS_DIR
    / "mlp_tuned_20170131_state_dict.pt"
)

META_PATH = (
    MODELS_DIR
    / "mlp_tuned_20170131_meta.json"
)

HISTORY_PATH = (
    MODELS_DIR
    / "mlp_tuned_20170131_history.csv"
)

PREDICTION_PATH = (
    PROCESSED_DIR
    / "mlp_tuned_test_predictions_20170131.csv"
)

# 최적 Epoch의 모델 가중치 저장
torch.save(
    final_experiment["model"].state_dict(),
    MODEL_PATH,
)

# Epoch별 학습 기록 저장
final_experiment["history"].to_csv(
    HISTORY_PATH,
    index=False,
)

# Test 고객별 예측 결과 저장
mlp_test_predictions = test_df[
    ["msno", "split", "is_churn"]
].copy()

mlp_test_predictions[
    "churn_probability"
] = test_probabilities

mlp_test_predictions[
    "predicted_churn"
] = test_predictions

mlp_test_predictions.to_csv(
    PREDICTION_PATH,
    index=False,
)

test_confusion_matrix = confusion_matrix(
    test_targets,
    test_predictions,
)

metadata = {
    "model_name": "MLP tuned",
    "snapshot": "2017-01-31",
    "target": "is_churn",
    "input_feature_count": int(
        len(FINAL_FEATURE_NAMES)
    ),
    "hidden_layers": [128, 64],
    "activation": "ReLU",
    "dropout": [0.3, 0.2],
    "batch_normalization": True,
    "optimizer": "AdamW",
    "learning_rate": float(
        LEARNING_RATE
    ),
    "weight_decay": float(
        WEIGHT_DECAY
    ),
    "batch_size": int(BATCH_SIZE),
    "pos_weight": float(
        final_experiment["pos_weight"]
    ),
    "best_epoch": int(
        final_experiment["best_epoch"]
    ),
    "threshold_source": (
        "Validation F1 maximum"
    ),
    "threshold": float(final_threshold),
    "valid_auc": float(
        final_experiment[
            "valid_result"
        ]["auc"]
    ),
    "valid_pr_auc": float(
        final_experiment[
            "valid_result"
        ]["pr_auc"]
    ),
    "test_auc": float(
        test_result["auc"]
    ),
    "test_pr_auc": float(
        test_result["pr_auc"]
    ),
    "test_precision": float(
        precision_score(
            test_targets,
            test_predictions,
        )
    ),
    "test_recall": float(
        recall_score(
            test_targets,
            test_predictions,
        )
    ),
    "test_f1": float(
        f1_score(
            test_targets,
            test_predictions,
        )
    ),
    "test_confusion_matrix": (
        test_confusion_matrix.tolist()
    ),
    "preprocessor_path": (
        "mlp_20170131_preprocessor.joblib"
    ),
}

with open(
    META_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("모델:", MODEL_PATH)
print("메타데이터:", META_PATH)
print("학습 기록:", HISTORY_PATH)
print("Test 예측:", PREDICTION_PATH)

print("\n최종 MLP 저장 완료")

모델: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\models\mlp_tuned_20170131_state_dict.pt
메타데이터: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\models\mlp_tuned_20170131_meta.json
학습 기록: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\models\mlp_tuned_20170131_history.csv
Test 예측: C:\Users\MoonSungHo\skn34-st\team_project\SKN34-2nd-2Team-modify\data\processed\mlp_tuned_test_predictions_20170131.csv

최종 MLP 저장 완료
